# Small Models, Real Constraints

Runs every experiment in the project and produces every table and figure. Everything runs on a CPU.

1. In the first code cell choose `QUICK = True` (checks the pipeline on a 20,000-document subsample) or `QUICK = False` (the full run, about two hours).
2. Run every cell from top to bottom.
3. The last cell zips the results and figures for download.

On Kaggle, turn **Internet** on in the side panel. If a full run is interrupted, re-run the clone and install cells, then use `--from <stage>` in the run cell to resume.

In [ ]:
QUICK = True   # set to False for the full run

GITHUB_URL = "https://github.com/MosaabSaleem/Capstone"

import os, shutil, subprocess
BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else ("/content" if os.path.isdir("/content") else os.getcwd())
os.chdir(BASE)
target = os.path.join(BASE, "Capstone")
shutil.rmtree(target, ignore_errors=True)
result = subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, target], capture_output=True, text=True)
if result.returncode != 0:
    print("git clone failed:", result.stderr)
    for root, dirs, files in os.walk("/kaggle/input"):
        if "requirements.txt" in files and "src" in dirs:
            shutil.copytree(root, target); print("Using the copy attached at", root); break

project = next((d for d, dirs, files in os.walk(target) if "requirements.txt" in files and "src" in dirs), None)
assert project, "Could not find the project. Check GITHUB_URL or attach the repo as a dataset."
os.chdir(project)
FLAGS = "--quick" if QUICK else ""
RESULTS = "results_quick" if QUICK else "results"
FIGURES = "figures_quick" if QUICK else "figures"
print("Project folder:", os.getcwd(), "| mode:", "quick check" if QUICK else "full run")

## 1. Install

In [ ]:
!pip install -q -r requirements.txt

## 2. Smoke test
Runs the whole pipeline on a tiny synthetic dataset. It should end with `SMOKE TEST PASSED`.

In [ ]:
!python scripts/smoke_test.py

## 3. Run every experiment
To resume an interrupted run, add `--from <stage>` (the error message names the stage).

In [ ]:
!python scripts/run_all.py {FLAGS}

## 4. Figures

In [ ]:
import glob
from IPython.display import Image, display
for path in sorted(glob.glob(f"{FIGURES}/*.png")):
    print(os.path.basename(path)); display(Image(filename=path))

## 5. Download the results

In [ ]:
import zipfile
archive = os.path.join(BASE, "capstone_outputs.zip")
with zipfile.ZipFile(archive, "w") as zf:
    for folder in (RESULTS, FIGURES):
        for path in glob.glob(f"{folder}/*"):
            zf.write(path)
print("Created", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("On Kaggle, download capstone_outputs.zip from the Output panel.")